In [ ]:

import os
import subprocess
import time
from pathlib import Path

started = time.monotonic()
output = Path('/kaggle/working')
work = Path('/kaggle/tmp/fcos')
work.mkdir(parents=True, exist_ok=True)
with (output / 'gpu.txt').open('w') as record:
    subprocess.run(['nvidia-smi'], check=True, stdout=record)
miniforge = work / 'miniforge.sh'
subprocess.run(['curl', '-fLsS', '--retry', '3', '-o', str(miniforge),
                'https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh'],
               check=True, timeout=600)
conda = work / 'conda'
subprocess.run(['bash', str(miniforge), '-b', '-p', str(conda)], check=True, timeout=900)
prefix = work / 'py310'
subprocess.run([str(conda / 'bin/conda'), 'create', '-y', '-p', str(prefix), 'python=3.10'],
               check=True, timeout=1200)
py = str(prefix / 'bin/python')
pip = [py, '-m', 'pip', 'install', '--disable-pip-version-check']
subprocess.run(pip + ['setuptools<70', 'numpy==1.26.4', 'opencv-python==4.10.0.84',
                       'pycocotools==2.0.10', 'yapf==0.40.1'], check=True, timeout=900)
subprocess.run(pip + ['torch==2.0.1+cu118', 'torchvision==0.15.2+cu118',
                       '--index-url', 'https://download.pytorch.org/whl/cu118'], check=True, timeout=1500)
subprocess.run(pip + ['mmcv-full==1.7.2', '--only-binary=mmcv-full', '-f',
                       'https://download.openmmlab.com/mmcv/dist/cu118/torch2.0/index.html'],
               check=True, timeout=900)
subprocess.run(pip + ['mmdet==2.28.2'], check=True, timeout=900)
with (output / 'pip-freeze.txt').open('w') as record:
    subprocess.run([py, '-m', 'pip', 'freeze'], check=True, stdout=record)
repo = work / 'mmdetection'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'v2.28.2',
                'https://github.com/open-mmlab/mmdetection.git', str(repo)], check=True, timeout=300)


In [ ]:

import hashlib
import os
import subprocess
import time
from pathlib import Path

source = "\nimport copy\nimport hashlib\nimport json\nimport subprocess\nfrom collections import Counter\nfrom pathlib import Path\nimport time\n\nimport torch\nimport mmcv\nfrom mmcv.ops import nms\nfrom mmcv.parallel import scatter\nfrom mmdet.apis import init_detector\nfrom mmdet.core import bbox2result\nfrom mmdet.datasets import build_dataloader, build_dataset\n\noutput = Path('/kaggle/working')\nwork = Path('/kaggle/tmp/fcos')\nrepo = work / 'mmdetection'\nconfig_rel = 'configs/fcos/fcos_r50_caffe_fpn_gn-head_1x_coco.py'\ncheckpoint_name = 'fcos_r50_caffe_fpn_gn-head_1x_coco-821213aa.pth'\ncheckpoint_url = ('https://download.openmmlab.com/mmdetection/v2.0/fcos/'\n                  'fcos_r50_caffe_fpn_gn-head_1x_coco/' + checkpoint_name)\n\n\ndef sha256(path):\n    with path.open('rb') as stream:\n        return hashlib.file_digest(stream, 'sha256').hexdigest() if hasattr(hashlib, 'file_digest') else stream_sha256(stream)\n\n\ndef stream_sha256(stream):\n    digest = hashlib.sha256()\n    for chunk in iter(lambda: stream.read(1024 * 1024), b''):\n        digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef log(record):\n    line = json.dumps(record, ensure_ascii=False)\n    with (output / 'metrics.jsonl').open('a', encoding='utf-8') as stream:\n        stream.write(line + '\\n')\n    print(line, flush=True)\n\n\ndef box_set(detections, labels):\n    return Counter((label, *box[:4]) for box, label in zip(detections.tolist(), labels.tolist()))\n\n\ndef run(deadline, prereg_commit, notebook_sha256):\n    torch.manual_seed(0)\n    torch.backends.cudnn.benchmark = False\n    assert torch.cuda.is_available(), 'CUDA unavailable'\n    boxes = torch.tensor([[0., 0., 10., 10.], [1., 1., 10., 10.]], device='cuda:0')\n    _, kept = nms(boxes, torch.tensor([0.9, 0.8], device='cuda:0'), 0.5)\n    assert kept.tolist() == [0], 'CUDA NMS smoke failed'\n    log(dict(stage='cuda_nms_ok', gpu=torch.cuda.get_device_name(0), cuda=torch.version.cuda,\n             prereg_commit=prereg_commit, notebook_sha256=notebook_sha256))\n\n    ann_candidates = sorted(Path('/kaggle/input').rglob('instances_val2017.json'))\n    image_candidates = sorted(Path('/kaggle/input').rglob('000000000139.jpg'))\n    assert len(ann_candidates) == 1, f'Expected one COCO annotation file, found {ann_candidates}'\n    image_dirs = [path.parent for path in image_candidates if path.parent.name == 'val2017']\n    assert len(image_dirs) == 1, f'Expected one COCO val2017 image directory, found {image_dirs}'\n    ann_file = ann_candidates[0]\n    image_dir = image_dirs[0]\n    annotations = json.loads(ann_file.read_text(encoding='utf-8'))\n    image_ids = {image['id'] for image in annotations['images']}\n    assert len(annotations['images']) == len(image_ids) == 5000\n    assert len(annotations['annotations']) == 36781 and len(annotations['categories']) == 80\n    assert all((image_dir / image['file_name']).is_file() for image in annotations['images'])\n    log(dict(stage='data_ok', images=5000, instances=36781, ann_sha256=sha256(ann_file)))\n\n    weights = work / checkpoint_name\n    subprocess.run(['curl', '-fLsS', '--retry', '3', '-o', str(weights), checkpoint_url],\n                   check=True, timeout=600)\n    weights_hash = sha256(weights)\n    assert weights_hash.startswith('821213aa'), f'Checkpoint SHA256 mismatch: {weights_hash}'\n    cfg = mmcv.Config.fromfile(repo / config_rel)\n    cfg.model.backbone.init_cfg = None\n    detector = init_detector(cfg, device='cpu')\n    state = torch.load(weights, map_location='cpu')['state_dict']\n    detector.load_state_dict(state, strict=True)\n    detector = detector.eval().to('cuda:0')\n    cfg.dump(str(output / 'resolved-config.py'))\n    log(dict(stage='weights_ok', weights_sha256=weights_hash, weights_bytes=weights.stat().st_size,\n             weights_url=checkpoint_url, config_sha256=sha256(output / 'resolved-config.py'),\n             mmdet_commit=subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()))\n\n    test_cfg = copy.deepcopy(cfg.data.test)\n    test_cfg.ann_file = str(ann_file)\n    test_cfg.img_prefix = str(image_dir)\n    test_cfg.test_mode = True\n    dataset = build_dataset(test_cfg)\n    assert len(dataset) == 5000 and set(dataset.img_ids) == image_ids\n    ids_bytes = json.dumps(dataset.img_ids, separators=(',', ':')).encode()\n    log(dict(stage='split_ok', ordered_image_ids_sha256=hashlib.sha256(ids_bytes).hexdigest()))\n    loader = build_dataloader(dataset, samples_per_gpu=1, workers_per_gpu=2, dist=False, shuffle=False)\n    head = detector.bbox_head\n    full_cfg = copy.deepcopy(head.test_cfg)\n    limit = full_cfg.max_per_img\n    assert limit == 100 and full_cfg.nms_pre == 1000\n    full_cfg.max_per_img = 10000\n    native_results, cls_results = [], []\n    changed_nms = changed_final = 0\n    with torch.inference_mode(), (output / 'paired-box-counts.csv').open('w', encoding='utf-8') as rows:\n        rows.write('image_id,native_nms,cls_nms,nms_set_equal,final_set_equal\\n')\n        for index, batch in enumerate(loader):\n            if time.monotonic() >= deadline:\n                log(dict(stage='partial_time_limit', images=index, changed_nms=changed_nms, changed_final=changed_final))\n                return\n            sample = scatter(batch, [0])[0]\n            image, metas = sample['img'][0], sample['img_metas'][0]\n            cls, reg, center = head(detector.extract_feat(image))\n            a, labels_a = head.get_bboxes(cls, reg, center, img_metas=metas, cfg=full_cfg, rescale=True)[0]\n            b, labels_b = head.get_bboxes(cls, reg, None, img_metas=metas, cfg=full_cfg, rescale=True)[0]\n            assert len(a) <= 5 * full_cfg.nms_pre and len(b) <= 5 * full_cfg.nms_pre\n            capped_a = bbox2result(a[:limit], labels_a[:limit], head.num_classes)\n            capped_b = bbox2result(b[:limit], labels_b[:limit], head.num_classes)\n            if index < 8:\n                import numpy as np\n                reference = detector.simple_test(image, metas, rescale=True)[0]\n                assert all(np.array_equal(left, right) for left, right in zip(capped_a, reference)), 'Native parity gate failed'\n                if index == 7:\n                    log(dict(stage='native_parity_ok', images=8))\n            same_nms = box_set(a, labels_a) == box_set(b, labels_b)\n            same_final = box_set(a[:limit], labels_a[:limit]) == box_set(b[:limit], labels_b[:limit])\n            changed_nms += not same_nms\n            changed_final += not same_final\n            rows.write(f'{dataset.img_ids[index]},{len(a)},{len(b)},{int(same_nms)},{int(same_final)}\\n')\n            native_results.append(capped_a)\n            cls_results.append(capped_b)\n            if (index + 1) % 250 == 0:\n                rows.flush()\n                log(dict(stage='progress', images=index + 1, changed_nms=changed_nms, changed_final=changed_final))\n\n    assert len(native_results) == len(cls_results) == 5000\n    native = dataset.evaluate(native_results, metric='bbox')\n    log(dict(stage='native_ap', metrics=native))\n    cls_only = dataset.evaluate(cls_results, metric='bbox')\n    log(dict(stage='cls_only_ap', metrics=cls_only))\n    from decimal import Decimal\n    native_ap = Decimal(str(native['bbox_mAP'])) * 100\n    delta = Decimal(str(cls_only['bbox_mAP'])) * 100 - native_ap\n    log(dict(stage='complete', images=5000, changed_nms=changed_nms, changed_final=changed_final,\n             baseline_in_band=Decimal('36.1') <= native_ap <= Decimal('37.1'),\n             cls_minus_native_ap_pp=float(delta)))\n"
prereg_commit = 'bd9dab93ef7c623d2f16e51799c9f833cd972c4f'
notebook_file = Path('/kaggle/working/__notebook__.ipynb')
notebook_sha256 = hashlib.sha256(notebook_file.read_bytes()).hexdigest() if notebook_file.exists() else None
remaining = int(5.5 * 3600 - (time.monotonic() - started))
assert remaining > 120, 'Setup consumed the available session budget'
entrypoint = work / 'evaluate.py'
entrypoint.write_text(source + '\nrun(' + repr(started + 5.5 * 3600 - 120) + ', ' + repr(prereg_commit) + ', ' + repr(notebook_sha256) + ')\n', encoding='utf-8')
subprocess.run([py, str(entrypoint)], env={**os.environ, 'CUDA_VISIBLE_DEVICES': '0'}, check=True, timeout=remaining)
